# P1B pre-ban calibration (reads data up to 3 November 2023 only)

P1B asks whether Korea's short-selling ban of 6 November 2023 changed each stock's price dynamics in proportion to
the short sellers' own feedback kernel, as Paper 1's sufficiency theorem predicts. Before the test can be registered,
its simulation check must be calibrated to real data, and only **pre-ban** data may be read for that.

This notebook:
1. downloads, for 1 June 2021 to 3 November 2023 only, daily prices, volumes and short-selling volumes of the
   KOSPI 200 and KOSDAQ 150 stocks that were constituents throughout, and of size-matched non-constituents;
2. fixes the treated and control lists (saved, with a hash);
3. computes the simulator's parameters (`p1b_core.calibrate`).

A guard refuses any request dated after 3 November 2023, so no ban-period data can be read by accident.

**Before running:** Colab Secrets `KRX_ID` and `KRX_PW` (key icon on the left, notebook access on), as for P1K.
Runtime: about 15 to 30 minutes. The download is cached in your Google Drive (`MyDrive/P1B_preban`), so a
disconnect resumes where it stopped. When it finishes, reply "P1B calibration done"; the result file
`MyDrive/P1B_preban/P1B_calibration.json` is read from Drive. The last cell also prints it.

Embedded estimator: `p1b_core.py`, sha256 `7fdb3d534e3df8dd5ce92c771ca5cd7dc89e1661f18ac5839b729ea14a5dca99`.


In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pykrx==1.2.9', 'pyarrow'], check=True)
from google.colab import userdata, drive
os.environ['KRX_ID'] = userdata.get('KRX_ID')
os.environ['KRX_PW'] = userdata.get('KRX_PW')
drive.mount('/content/drive')


In [ ]:
import hashlib, pathlib
CORE = '"""P1B: estimators and simulator for the Korea short-selling ban test (draft, not registered).\n\nOne module serves both the simulation check and, if the design passes and is registered, the data run, so the\nstatistic the simulation evaluates is the statistic the data run computes.\n\nPanel layout: arrays of shape (N stocks, T trading days). NaN marks a day without a valid observation.\n  r  daily log return of the stock          m  daily log return of the stock\'s market index (same shape)\n  s  short-selling share of volume (short volume / total volume); NaN when short selling is not allowed\nWindows are slices of the day axis.\n"""\nimport numpy as np\n\nLAGS = 5            # outcome: sum of the first 5 autocorrelations; kernel regressor: sum of the last 5 returns\nWINSOR = (0.5, 99.5)\nMIN_VALID = 0.80    # a stock-window needs at least 80% of the window\'s trading days valid\nVOL_SCALE = True    # standardize each day by a volatility forecast from earlier days (a weighted estimator)\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# Stock-level statistics\n# ----------------------------------------------------------------------------------------------------------------\ndef _winsor_rows(x):\n    lo, hi = np.nanpercentile(x, WINSOR, axis=1, keepdims=True)\n    return np.clip(x, lo, hi)\n\n\ndef standardized_residual(r, m, w, sector=None):\n    """Market- and sector-adjusted, winsorized, standardized daily return in window w (slice). Returns (z, valid_share).\n\n    valid_share counts days with a return and a market return (before the sector step).\n    """\n    r, m = r[:, w], m[:, w]\n    ok = np.isfinite(r) & np.isfinite(m)\n    n = ok.sum(1)\n    rr, mm = np.where(ok, r, 0.0), np.where(ok, m, 0.0)\n    with np.errstate(invalid="ignore", divide="ignore"):\n        rbar, mbar = rr.sum(1) / n, mm.sum(1) / n\n        cov = (ok * (rr - rbar[:, None]) * (mm - mbar[:, None])).sum(1) / n\n        var = (ok * (mm - mbar[:, None]) ** 2).sum(1) / n\n        beta = cov / var\n    e = np.where(ok, r - rbar[:, None] - beta[:, None] * (m - mbar[:, None]), np.nan)\n    e = _winsor_rows(e)\n    if sector is not None:  # subtract the leave-one-out mean of the stock\'s sector (sectors with >= 3 stocks)\n        for sec in np.unique(sector):\n            idx = np.where(sector == sec)[0]\n            if idx.size < 3:\n                continue\n            blk = e[idx]\n            cnt = np.isfinite(blk).sum(0)\n            tot = np.nansum(blk, 0)\n            with np.errstate(invalid="ignore", divide="ignore"):\n                loo = (tot[None, :] - np.nan_to_num(blk)) / (cnt[None, :] - np.isfinite(blk))\n            e[idx] = np.where(cnt[None, :] - np.isfinite(blk) >= 2, blk - loo, np.nan)\n    if VOL_SCALE:  # divide by a volatility forecast built from earlier days only (EWMA, decay 0.94)\n        e = e / _ewma_vol(e)\n        e = _winsor_rows(e)\n    e = e - np.nanmean(e, 1, keepdims=True)\n    z = e / np.nanstd(e, 1, keepdims=True)\n    return z, n / r.shape[1]\n\n\ndef _ewma_vol(e, lam=0.94, warm=20):\n    """sigma_t from e_{t-1}, e_{t-2}, ...; started at the variance of the first `warm` valid days of the window."""\n    N, T = e.shape\n    out = np.full((N, T), np.nan)\n    v = np.array([np.nanvar(row[np.isfinite(row)][:warm]) if np.isfinite(row).sum() >= 5 else np.nan for row in e])\n    for t in range(T):\n        out[:, t] = np.sqrt(v)\n        x = e[:, t]\n        v = np.where(np.isfinite(x), lam * v + (1 - lam) * x ** 2, v)\n    return out\n\n\ndef autocorr_sum(z, lags=LAGS):\n    """A = sum_{j=1..lags} rho_j, rho_j = sum z_t z_{t-j} / sum z_t^2 over days where both are valid."""\n    den = np.nansum(z * z, 1)\n    a = np.zeros(z.shape[0])\n    for j in range(1, lags + 1):\n        a += np.nansum(z[:, j:] * z[:, :-j], 1) / den\n    return a\n\n\ndef past_sum(z, lags=LAGS):\n    """R_{t-1} = z_{t-1} + ... + z_{t-lags}; NaN unless all are valid."""\n    out = np.full_like(z, np.nan)\n    acc = np.zeros((z.shape[0], z.shape[1] - lags))\n    for j in range(1, lags + 1):\n        acc = acc + z[:, lags - j: z.shape[1] - j]\n    out[:, lags:] = acc\n    return out\n\n\ndef shorting_kernel(s, z):\n    """gamma_i: OLS coefficient of the shorting share s_t on (1, R_{t-1}): the class\'s response to past prices only.\n\n    Positive gamma means short sellers short more after the stock has risen (contrarian). The same-day return is not\n    a regressor: it is moved by the same day\'s short sales, so controlling for it would bias gamma.\n    Returns (gamma, se_iid, nobs). The standard error is reported only; no decision uses it.\n    """\n    R = past_sum(z)\n    s = _winsor_rows(s)\n    ok = np.isfinite(s) & np.isfinite(R)\n    X = np.stack([np.ones_like(z), np.where(ok, R, 0)], -1) * ok[..., None]\n    y = np.where(ok, s, 0.0)\n    XtX = np.einsum("ntk,ntl->nkl", X, X)\n    Xty = np.einsum("ntk,nt->nk", X, y)\n    n = ok.sum(1)\n    good = n >= 30\n    b = np.full((z.shape[0], 2), np.nan)\n    b[good] = np.linalg.solve(XtX[good], Xty[good][..., None])[..., 0]\n    res = y - np.einsum("ntk,nk->nt", X, np.nan_to_num(b))\n    s2 = (ok * res ** 2).sum(1) / np.maximum(n - 2, 1)\n    se = np.full(z.shape[0], np.nan)\n    se[good] = np.sqrt(s2[good] * np.linalg.inv(XtX[good])[:, 1, 1])\n    return b[:, 1], se, n\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# Cross-sectional regressions\n# ----------------------------------------------------------------------------------------------------------------\ndef ols_hc3(y, X):\n    """OLS with HC3 standard errors. Returns (coef, se)."""\n    ok = np.isfinite(y) & np.isfinite(X).all(1)\n    y, X = y[ok], X[ok]\n    XtXi = np.linalg.inv(X.T @ X)\n    b = XtXi @ X.T @ y\n    e = y - X @ b\n    h = np.einsum("ij,jk,ik->i", X, XtXi, X)\n    u = e / (1 - h)\n    V = XtXi @ (X.T * u ** 2) @ X @ XtXi\n    return b, np.sqrt(np.diag(V)), ok.sum()\n\n\ndef _z(x):\n    return (x - np.nanmean(x)) / np.nanstd(x)\n\n\ndef p_one_sided(t, sign):\n    """One-sided p-value for H1: coefficient has the given sign (normal approximation)."""\n    from math import erfc, sqrt\n    return 0.5 * erfc(sign * t / sqrt(2))\n\n\ndef decide(stats, alpha=0.05):\n    """Registered verdict.\n\n    Primary statistic K: slope theta of D_i = A_i(ban) - [A_i(before) + A_i(after)]/2 on the standardized pre-ban\n    shorting kernel gamma_i (treated stocks). Theory: removing a contrarian class (gamma > 0) raises A in\n    proportion to gamma, and its return lowers A again, so theta > 0. K2 (ban only) and K3 (reinstatement only)\n    must agree in sign for a supportive verdict.\n    """\n    pK = p_one_sided(stats["t_K"], +1)\n    if p_one_sided(stats["t_K"], -1) < alpha:\n        return "contradicted"\n    if pK < alpha and stats["theta_K2"] > 0 and stats["theta_K3"] < 0:\n        return "supportive"\n    return "inconclusive"\n\n\ndef run_statistics(panel, W):\n    """All registered statistics from a panel dict and window dict. Uses no data outside the windows in W."""\n    r, m, s = panel["r"], panel["m"], panel["s"]\n    treated, kosdaq = panel["treated"], panel["kosdaq"]\n    logcap, logturn = panel["logcap"], panel["logturn"]\n    sec = panel.get("sector")\n    z0a, v0a = standardized_residual(r, m, W["W0a"], sec)\n    z0b, v0b = standardized_residual(r, m, W["W0b"], sec)\n    z1, v1 = standardized_residual(r, m, W["W1"], sec)\n    z2, v2 = standardized_residual(r, m, W["W2"], sec)\n    gam, gse, _ = shorting_kernel(s[:, W["W0a"]], z0a)\n    A0, A1, A2 = autocorr_sum(z0b), autocorr_sum(z1), autocorr_sum(z2)\n    with np.errstate(all="ignore"):\n        sbar = np.nanmean(np.where(np.isfinite(s[:, W["W0a"]]), s[:, W["W0a"]], np.nan), 1)\n    valid = (v0a >= MIN_VALID) & (v0b >= MIN_VALID) & (v1 >= MIN_VALID) & (v2 >= MIN_VALID)\n    T = valid & treated & np.isfinite(gam) & np.isfinite(sbar)\n    C = valid & ~treated\n    out = {"n_treated": int(T.sum()), "n_control": int(C.sum())}\n    g = _z(np.where(T, gam, np.nan))[T]\n    ctrl = [_z(np.where(T, sbar, np.nan))[T], logcap[T], logturn[T], kosdaq[T].astype(float)]\n    X = np.column_stack([np.ones(T.sum()), g] + ctrl)\n    for key, y in [("K", A1 - (A0 + A2) / 2), ("K2", A1 - A0), ("K3", A2 - A1)]:\n        b, se, _ = ols_hc3(y[T], X)\n        out.update({f"theta_{key}": b[1], f"se_{key}": se[1], f"t_{key}": b[1] / se[1]})\n        if key == "K":\n            out.update(sbar_K5=b[2], t_K5=b[2] / se[2])\n    # K4 (reported): equal magnitude in the two events, theta_K2 + theta_K3 = 0 (conservative s.e.)\n    out["t_K4"] = (out["theta_K2"] + out["theta_K3"]) / np.hypot(out["se_K2"], out["se_K3"])\n    # K6 (reported): does gamma add to the price-only predictor A(before)?  Theory: yes (class tags are needed)\n    X6 = np.column_stack([X, _z(np.where(T, A0, np.nan))[T]])\n    b6, se6, _ = ols_hc3((A1 - A0)[T], X6)\n    out.update(theta_K6=b6[1], t_K6=b6[1] / se6[1])\n    # K1 (reported): difference in differences, treated against controls, with covariates\n    TC = T | C\n    XD = np.column_stack([np.ones(TC.sum()), treated[TC].astype(float), logcap[TC], logturn[TC],\n                          kosdaq[TC].astype(float)])\n    bd, sed, _ = ols_hc3((A1 - A0)[TC], XD)\n    gbar, gsd = np.nanmean(gam[T]), np.nanstd(gam[T]) / np.sqrt(T.sum())\n    out.update(did_K1=bd[1], t_K1=bd[1] / sed[1], gamma_mean=gbar, t_gamma_mean=gbar / gsd,\n               gamma_sd=np.nanstd(gam[T]), gamma_se_median=np.nanmedian(gse[T]))\n    out = {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in out.items()}\n    out["verdict"] = decide(out)\n    return out\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# Simulator\n# ----------------------------------------------------------------------------------------------------------------\nPRIOR = dict(  # literature-informed prior values; replaced by the pre-ban calibration before the official runs\n    n_treated=300, n_control=300,\n    days=dict(W0a=300, W0b=290, gap1=20, W1=330, gap2=22, W2=360, burn=120),\n    sbar_logmu=-3.12, sbar_logsd=0.5,       # log of a stock\'s mean shorting share of volume (mean about 5%)\n    gamma_mean=0.002, gamma_sd=0.002,       # shorting-share response per unit of R (R has s.d. about sqrt(5))\n    v_cv=0.6, v_ar=0.5,                     # shorting-share noise, s.d. relative to the stock\'s mean; persistence\n    same_day=0.003,                         # same-day response of shorting share to z_t (not a price effect)\n    phi_sd=0.01,                            # other classes\' feedback per lag (A_other has s.d. about 5*phi_sd)\n    garch=(0.08, 0.90), t_df=5,\n    sector_n=20, sector_share=0.15, gamma_sector_share=0.3,\n    halt_rate=0.01, long_halt_share=0.02,\n)\n\n\ndef simulate(rng, P, kappa, confound=0.0, shift_sd=0.0, lag_profile="flat"):\n    """One simulated panel. kappa scales the price effect of the shorting class\'s flow (kappa=0: no effect).\n\n    confound: post-ban change in other classes\' feedback per lag that loads on log size, with gamma also loading\n      on size (correlation 0.4), so a size-driven change looks like a gamma effect unless controls remove it.\n    shift_sd: s.d. of a common post-ban shift for treated stocks (affects K1 only).\n    lag_profile: \'flat\' (kernel spread over lags 1-5) or \'lag1\' (all on lag 1) or \'late\' (lags 6-10; outside A).\n    """\n    d = P["days"]\n    NT, NC = P["n_treated"], P["n_control"]\n    N = NT + NC\n    order = ["burn", "W0a", "W0b", "gap1", "W1", "gap2", "W2"]\n    edges = np.cumsum([0] + [d[k] for k in order])\n    T = edges[-1]\n    W = {k: slice(edges[i], edges[i + 1]) for i, k in enumerate(order)}\n    ban, rein = edges[order.index("gap1")], edges[order.index("gap2")]\n    treated = np.arange(N) < NT\n    kosdaq = rng.random(N) < 0.4\n    logcap = np.where(treated, rng.normal(0.6, 0.8, N), rng.normal(-0.6, 0.8, N))\n    logturn = 0.5 * logcap + rng.normal(0, 0.8, N)\n    sector = rng.integers(0, P["sector_n"], N)\n    sec_eff = rng.normal(0, 1, P["sector_n"])[sector]\n    zc = (logcap - logcap.mean()) / logcap.std()\n    gamma = P["gamma_mean"] + P["gamma_sd"] * (np.sqrt(P["gamma_sector_share"]) * sec_eff\n                                               + 0.4 * zc * np.sqrt(1 - P["gamma_sector_share"])\n                                               + np.sqrt(max(0.0, 0.84 - P["gamma_sector_share"])) * rng.normal(0, 1, N))\n    sbar = np.exp(rng.normal(P["sbar_logmu"], P["sbar_logsd"], N))\n    v_sd = P["v_cv"] * sbar\n    phi = rng.normal(0, P["phi_sd"], N)\n    dphi_ban = confound * zc + (rng.normal(0, shift_sd) if shift_sd else 0.0) * treated\n    dphi_rein = -confound * zc\n    beta = rng.uniform(0.5, 1.5, N)\n    mk = rng.normal(0, 1, T) * 0.8\n    secshock = rng.standard_t(P["t_df"], (P["sector_n"], T)) / np.sqrt(P["t_df"] / (P["t_df"] - 2))\n    a_g, b_g = P["garch"]\n    h = np.ones(N)\n    eps_prev = np.zeros(N)\n    u = np.zeros((N, T))                         # idiosyncratic return in units of the stock\'s s.d.\n    v = np.zeros(N)\n    s = np.full((N, T), np.nan)\n    if lag_profile == "flat":\n        lw = np.r_[np.ones(LAGS), np.zeros(5)]\n    elif lag_profile == "lag1":\n        lw = np.r_[LAGS, np.zeros(9)]\n    else:\n        lw = np.r_[np.zeros(5), np.ones(5)]\n    for t in range(T):\n        active = np.where(treated, (t < ban) | (t >= rein), t >= rein)\n        h = (1 - a_g - b_g) + a_g * eps_prev ** 2 * h + b_g * h\n        eps = rng.standard_t(P["t_df"], N) / np.sqrt(P["t_df"] / (P["t_df"] - 2))\n        noise = np.sqrt(h) * eps * np.sqrt(1 - P["sector_share"]) + np.sqrt(P["sector_share"]) * secshock[sector, t]\n        past = u[:, max(0, t - 10):t][:, ::-1]\n        k = past.shape[1]\n        Rk = past @ lw[:k] if k else np.zeros(N)\n        R5 = past[:, :LAGS].sum(1) if k else np.zeros(N)\n        ph = phi + np.where(t >= ban, dphi_ban, 0) + np.where(t >= rein, dphi_rein, 0)\n        v = P["v_ar"] * v + np.sqrt(1 - P["v_ar"] ** 2) * v_sd * rng.normal(0, 1, N)\n        flow = gamma * Rk + v                    # the class\'s flow beyond its mean (shorting share units)\n        u[:, t] = ph * R5 - kappa * active * flow + noise\n        st = sbar + gamma * Rk + P["same_day"] * u[:, t] + v\n        s[:, t] = np.where(active, np.clip(st, 0, 1), np.nan)\n        eps_prev = eps\n    sd = np.exp(rng.normal(np.log(0.022), 0.3, N))\n    r = beta[:, None] * mk[None, :] * 0.01 + sd[:, None] * u\n    m = np.tile(mk * 0.01, (N, 1))\n    halt = rng.random((N, T)) < P["halt_rate"]\n    long = rng.random(N) < P["long_halt_share"]\n    start = rng.integers(0, T - 200, N)\n    for i in np.where(long)[0]:\n        halt[i, start[i]:start[i] + 150] = True\n    r[halt] = np.nan\n    s[halt] = np.nan\n    panel = dict(r=r, m=m, s=s, treated=treated, kosdaq=kosdaq, logcap=logcap, logturn=logturn, sector=sector)\n    Wuse = {k: W[k] for k in ["W0a", "W0b", "W1", "W2"]}\n    truth = dict(theta_per_sd=5 * kappa * np.std(gamma[treated]), gamma=gamma)\n    return panel, Wuse, truth\n\n\n# ----------------------------------------------------------------------------------------------------------------\n# Calibration of the simulator from pre-ban data only (windows W0a and W0b end before 6 November 2023)\n# ----------------------------------------------------------------------------------------------------------------\ndef _ac(x, k):\n    a, b = x[:, k:], x[:, :-k]\n    ok = np.isfinite(a) & np.isfinite(b)\n    a, b = np.where(ok, a, 0), np.where(ok, b, 0)\n    n = ok.sum(1)\n    with np.errstate(invalid="ignore", divide="ignore"):\n        ma, mb = a.sum(1) / n, b.sum(1) / n\n        c = (ok * (a - ma[:, None]) * (b - mb[:, None])).sum(1) / n\n        va = (ok * (a - ma[:, None]) ** 2).sum(1) / n\n        vb = (ok * (b - mb[:, None]) ** 2).sum(1) / n\n        return c / np.sqrt(va * vb)\n\n\ndef _garch_from_acf(ac1, ac5):\n    pers = float(np.clip((max(ac5, 1e-4) / max(ac1, 1e-4)) ** 0.25, 0.5, 0.995))\n    best, ab = 1e9, (0.05, pers - 0.05)\n    for al in np.linspace(0.01, min(0.3, pers - 0.01), 300):\n        be = pers - al\n        den = 1 - 2 * al * be - be ** 2\n        if den <= 0:\n            continue\n        rho1 = al * (1 - al * be - be ** 2) / den\n        if abs(rho1 - ac1) < best:\n            best, ab = abs(rho1 - ac1), (float(al), float(be))\n    return ab\n\n\ndef _df_from_kurtosis(garch, target, T, grid=(4.5, 5, 6, 7, 8, 10, 15, 30), n=400, seed=20261121):\n    rng = np.random.default_rng(seed)\n    al, be = garch\n    best, out = 1e9, 30.0\n    for df in grid:\n        h, x = np.ones(n), np.zeros((n, T + 100))\n        ep = np.zeros(n)\n        for t in range(T + 100):\n            h = (1 - al - be) + al * ep ** 2 * h + be * h\n            ep = rng.standard_t(df, n) / np.sqrt(df / (df - 2))\n            x[:, t] = np.sqrt(h) * ep\n        x = _winsor_rows(x[:, 100:])\n        k = np.nanmedian(np.mean(x ** 4, 1) / np.mean(x ** 2, 1) ** 2 - 3)\n        if abs(k - target) < best:\n            best, out = abs(k - target), float(df)\n    return out\n\n\ndef calibrate(panel, Wa, Wb, later_days=(329, 373)):\n    """Simulator parameters and descriptive statistics from the pre-ban windows Wa (kernel) and Wb (outcome).\n\n    Every quantity uses only the days in Wa and Wb. later_days are calendar estimates of the lengths of W1 and W2;\n    they are not read from data.\n    """\n    r, m, s, sec = panel["r"], panel["m"], panel["s"], panel.get("sector")\n    treated = panel["treated"]\n    za, va = standardized_residual(r, m, Wa, sec)\n    zb, vb = standardized_residual(r, m, Wb, sec)\n    valid = (va >= MIN_VALID) & (vb >= MIN_VALID)\n    sa = s[:, Wa]\n    gam, gse, _ = shorting_kernel(sa, za)\n    h = za.shape[1] // 2\n    g1, _, _ = shorting_kernel(sa[:, :h], za[:, :h])\n    g2, _, _ = shorting_kernel(sa[:, h:], za[:, h:])\n    with np.errstate(all="ignore"):\n        sbar = np.nanmean(sa, 1)\n    ok = valid & treated & np.isfinite(gam) & np.isfinite(g1) & np.isfinite(g2) & (sbar > 0)\n    cov12 = float(np.cov(g1[ok], g2[ok])[0, 1])\n    gamma_sd = float(np.sqrt(max(cov12, 0.0)))\n    if sec is not None and gamma_sd > 0:\n        m1, m2 = g1.copy(), g2.copy()\n        for q in np.unique(sec[ok]):\n            j = ok & (sec == q)\n            m1[j], m2[j] = g1[j].mean(), g2[j].mean()\n        gshare = float(np.clip(np.cov(m1[ok], m2[ok])[0, 1] / cov12, 0.0, 0.8))\n    else:\n        gshare = 0.0\n    with np.errstate(all="ignore"):\n        cv = np.nanstd(sa, 1) / sbar\n    Aa, Ab = autocorr_sum(za), autocorr_sum(zb)\n    phi_sd = float(np.sqrt(max(np.cov(Aa[ok], Ab[ok])[0, 1], 0.0)) / LAGS)\n    zz = zb[ok] ** 2\n    ac1, ac5 = float(np.nanmedian(_ac(zz, 1))), float(np.nanmedian(_ac(zz, 5)))\n    garch = _garch_from_acf(ac1, ac5)\n    # tails: choose the t degrees of freedom whose GARCH-t series, winsorized like the data, matches the median\n    # sample excess kurtosis of z in Wb\n    with np.errstate(all="ignore"):\n        kurt = np.nanmean(zb[ok] ** 4, 1) / np.nanmean(zb[ok] ** 2, 1) ** 2 - 3\n    kx = float(np.nanmedian(kurt))\n    t_df = _df_from_kurtosis(garch, kx, zb.shape[1])\n    # sector common share: mean within-sector correlation of market-adjusted (not sector-adjusted) residuals\n    zm, _ = standardized_residual(r, m, Wb, None)\n    cors, ns = [], 0\n    if sec is not None:\n        for q in np.unique(sec[valid]):\n            j = np.where(valid & (sec == q))[0]\n            if j.size < 3:\n                continue\n            ns += 1\n            C = np.ma.corrcoef(np.ma.masked_invalid(zm[j]))\n            cors.append((C.sum() - np.trace(C)) / (j.size * (j.size - 1)))\n    sector_share = float(np.clip(np.mean(cors), 0.0, 0.6)) if cors else 0.0\n    vt = (va + vb) / 2\n    sim = dict(\n        n_treated=int(ok.sum()), n_control=int((valid & ~treated).sum()),\n        days=dict(W0a=int(za.shape[1]), W0b=int(zb.shape[1]), W1=int(later_days[0]), W2=int(later_days[1])),\n        sbar_logmu=float(np.mean(np.log(sbar[ok]))), sbar_logsd=float(np.std(np.log(sbar[ok]))),\n        gamma_mean=float(np.mean(gam[ok])), gamma_sd=gamma_sd,\n        v_cv=float(np.nanmedian(cv[ok])), v_ar=float(np.nanmedian(_ac(sa[ok], 1))),\n        phi_sd=phi_sd, garch=list(garch), t_df=t_df,\n        sector_n=int(max(ns, 1)), sector_share=sector_share, gamma_sector_share=gshare,\n        halt_rate=float(1 - np.mean(vt[valid])), long_halt_share=float(1 - valid[treated].mean()),\n    )\n    # descriptive: the pre-ban cross-section of A(W0b) on gamma (theory: slope -theta, if the association is causal)\n    g = _z(np.where(ok, gam, np.nan))[ok]\n    X = np.column_stack([np.ones(ok.sum()), g, _z(np.where(ok, sbar, np.nan))[ok], panel["logcap"][ok],\n                         panel["logturn"][ok], panel["kosdaq"][ok].astype(float)])\n    b, se, _ = ols_hc3(Ab[ok], X)\n    rep = dict(gamma_mean_t=float(np.mean(gam[ok]) / (np.std(gam[ok]) / np.sqrt(ok.sum()))),\n               gamma_hat_sd=float(np.std(gam[ok])), gamma_split_half_cov=cov12,\n               gamma_se_median=float(np.nanmedian(gse[ok])), A_split_half_cov=float(np.cov(Aa[ok], Ab[ok])[0, 1]),\n               A0_mean=float(np.mean(Ab[ok])), A0_sd=float(np.std(Ab[ok])), zsq_ac1=ac1, zsq_ac5=ac5,\n               excess_kurtosis_median=kx, theta_pre=float(b[1]), t_theta_pre=float(b[1] / se[1]),\n               implied_theta_per_sd_if_causal=float(-b[1]))\n    return sim, rep\n'

pathlib.Path('/content/p1b_core.py').write_text(CORE)
assert hashlib.sha256(CORE.encode()).hexdigest() == '7fdb3d534e3df8dd5ce92c771ca5cd7dc89e1661f18ac5839b729ea14a5dca99', 'p1b_core differs from the repository'
print('p1b_core.py sha256', hashlib.sha256(CORE.encode()).hexdigest())


In [ ]:
import os, sys, json, time, hashlib
import numpy as np, pandas as pd
sys.path.insert(0, '/content')
import p1b_core as B
from pykrx import stock

FIRST, SPLIT, LAST = '20210601', '20220901', '20231103'   # W0a = [FIRST, SPLIT), W0b = [SPLIT, LAST]
CACHE = '/content/drive/MyDrive/P1B_preban'
for sub in ('ohlcv', 'short'):
    os.makedirs(f'{CACHE}/{sub}', exist_ok=True)

def guard(*dates):
    for d in dates:
        d = str(d).replace('-', '')[:8]
        if d > LAST:
            raise SystemExit(f'refused: {d} is after {LAST}. P1B calibration reads pre-ban data only.')

def need(df, cols, what):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise SystemExit(f'{what}: expected columns {cols}, got {list(df.columns)}. Stopped; send this message back.')

def pause():
    time.sleep(0.3)

# 1. Trading calendar and market returns
guard(FIRST, LAST)
idx = {}
for mk, code in (('KOSPI', '1001'), ('KOSDAQ', '2001')):
    df = stock.get_index_ohlcv_by_date(FIRST, LAST, code, name_display=False); pause()
    need(df, ['종가'], f'index {code}')
    idx[mk] = df['종가'].astype(float)
days = [d.strftime('%Y%m%d') for d in idx['KOSPI'].index]
assert days[0] >= FIRST and days[-1] <= LAST, (days[0], days[-1])
cal = pd.DatetimeIndex(idx['KOSPI'].index)
mret = {mk: np.log(s.reindex(cal)).diff().values for mk, s in idx.items()}
print('trading days', len(days), days[0], '-', days[-1])

# 2. Constituents throughout (KOSPI 200 = 1028, KOSDAQ 150 = 2203)
CHECK = ['20210601', '20211201', '20220602', '20221201', '20230601', '20231103']
def tday(d):
    return next(x for x in days if x >= d)
mem = {}
for mk, code in (('KOSPI', '1028'), ('KOSDAQ', '2203')):
    sets = []
    for d in CHECK:
        guard(d)
        sets.append(set(stock.get_index_portfolio_deposit_file(code, tday(d)))); pause()
    if min(len(s) for s in sets) < 100:
        raise SystemExit(f'constituent lists for {code} look wrong: sizes {[len(s) for s in sets]}')
    mem[mk] = (set.intersection(*sets), set.union(*sets))

# 3. Universe, size, sectors (as of 3 November 2023)
rows = []
for mk in ('KOSPI', 'KOSDAQ'):
    c0 = stock.get_market_cap_by_ticker(days[0], market=mk); pause()
    c1 = stock.get_market_cap_by_ticker(days[-1], market=mk); pause()
    need(c1, ['시가총액'], 'market cap')
    sec = stock.get_market_sector_classifications(days[-1], mk); pause()
    need(sec, ['종목명', '업종명'], 'sectors')
    for t in c0.index.intersection(c1.index).intersection(sec.index):
        name = str(sec.loc[t, '종목명'])
        common = t.endswith('0') and '스팩' not in name and '리츠' not in name
        rows.append(dict(ticker=t, market=mk, name=name, sector=mk + ':' + str(sec.loc[t, '업종명']),
                         cap=float(c1.loc[t, '시가총액']), common=common,
                         treated=t in mem[mk][0], ever=t in mem[mk][1]))
U = pd.DataFrame(rows)
U = U[U.common & (U.cap > 0)]
T_ = U[U.treated].sort_values('cap', ascending=False)
pool = {mk: U[(U.market == mk) & ~U.ever].copy() for mk in ('KOSPI', 'KOSDAQ')}
ctrl = []
for _, row in T_.iterrows():
    p = pool[row.market]
    if p.empty:
        continue
    j = (np.log(p.cap) - np.log(row.cap)).abs().idxmin()
    ctrl.append(p.loc[j]); pool[row.market] = p.drop(j)
C_ = pd.DataFrame(ctrl)
print('treated', len(T_), 'controls', len(C_))
lists = {'treated': T_[['ticker', 'market', 'sector']].values.tolist(),
         'control': C_[['ticker', 'market', 'sector']].values.tolist()}
lists_txt = json.dumps(lists, ensure_ascii=False, sort_keys=True)
lists_sha = hashlib.sha256(lists_txt.encode()).hexdigest()
open(f'{CACHE}/P1B_lists.json', 'w').write(lists_txt)

# 4. Per-stock downloads (cached)
def cached(kind, t, fn):
    f = f'{CACHE}/{kind}/{t}.parquet'
    if os.path.exists(f):
        return pd.read_parquet(f)
    guard(FIRST, LAST)
    df = fn(); pause()
    df.to_parquet(f)
    return df
both = pd.concat([T_, C_])
first = True
R = np.full((len(both), len(days)), np.nan); S = np.full_like(R, np.nan); M = np.full_like(R, np.nan)
turn = np.full(len(both), np.nan)
for i, (_, row) in enumerate(both.iterrows()):
    o = cached('ohlcv', row.ticker, lambda: stock.get_market_ohlcv_by_date(FIRST, LAST, row.ticker, adjusted=True))
    need(o, ['종가', '거래량', '거래대금'], 'ohlcv')
    o = o.reindex(cal)
    vol = o['거래량'].astype(float).values
    live = vol > 0
    lr = np.log(o['종가'].astype(float).where(live)).diff().values
    lr[1:][~live[:-1]] = np.nan
    R[i] = lr; M[i] = mret[row.market]
    turn[i] = np.nanmean(o['거래대금'].astype(float).values[:days.index(tday(SPLIT))]) / row.cap
    if row.treated:
        sh = cached('short', row.ticker, lambda: stock.get_shorting_volume_by_date(FIRST, LAST, row.ticker))
        if first:
            print('short-selling columns:', list(sh.columns)); first = False
        need(sh, ['공매도'], 'short selling')
        sh = sh.reindex(cal)
        S[i] = np.where(live, sh['공매도'].astype(float).values / np.where(live, vol, 1), np.nan)
    if i % 50 == 0:
        print(i, 'of', len(both), flush=True)

secs = {s: k for k, s in enumerate(sorted(both.sector.unique()))}
panel = dict(r=R, m=M, s=S, treated=both.treated.values.astype(bool), kosdaq=(both.market == 'KOSDAQ').values,
             logcap=np.log(both.cap.values), logturn=np.log(turn), sector=both.sector.map(secs).values)
k = days.index(tday(SPLIT))
sim, rep = B.calibrate(panel, slice(0, k), slice(k, len(days)))
out = dict(simulator=sim, report=rep, lists_sha256=lists_sha, n_treated_listed=len(T_), n_control_listed=len(C_),
           first_date_read=days[0], last_date_read=days[-1], core_sha256=hashlib.sha256(open('/content/p1b_core.py','rb').read()).hexdigest(),
           pykrx='1.2.9')
open(f'{CACHE}/P1B_calibration.json', 'w').write(json.dumps(out, indent=1))
print(json.dumps(out, indent=1))
print('\nSaved to MyDrive/P1B_preban/P1B_calibration.json. Last date read:', days[-1])
